# Phase 3: human review and the answer library
The Jev hybrid pipeline (run 4: 36/41 answerable questions auto-approved, 0 unsafe, $0.17 per run) gains the two things a real team needs:

1. **Human review, built into the graph.** Anything the pipeline isn't sure about pauses the graph with LangGraph's `interrupt()`, so a reviewer can **approve**, **edit** or **reject** it. The graph's state is saved to SQLite after every decision. You can stop half-way, come back later, and carry on exactly where you left off, without paying to answer anything twice.
2. **An answer library that learns.** Every answer a human approves or edits is stored with the policy sections and versions behind it. In later questionnaires, each question is checked against the library first. Jev judges whether the new question asks for *the same information* as an approved one. If it does, the approved answer is reused and **no Claude call is made**. If a policy section behind an approved answer has changed version since, that answer counts as stale and isn't reused.

```
question → Jev classify → library lookup ──hit──→ reuse approved answer
                                  └─miss─→ search + Jev gate → Sonnet draft → claim checks → score
all answers → human review (interrupt per question) → finalise → library grows → completed XLSX + audit
```

**Two changes to drafting, both from run 4:**
- Answers say *"the customer"*, never the prospect's name. That keeps approved answers reusable across prospects, and stops Jev reading *"Globex is the controller"* as contradicting *"the customer is the controller"*.
- Caveats for the reviewer go in a separate **reviewer note**, so wording like *"our policy doesn't say…"* never lands in customer-facing text.

**Before you start:** add `ANTHROPIC_API_KEY` and `TYPESAFE_API_KEY` in Colab Secrets. Have `acme-policy-pack.zip`, the questionnaire and `golden_set.csv` ready.

## 1. Setup and settings

In [ ]:
!pip install -q langgraph langchain-anthropic typesafe-sdk langgraph-checkpoint-sqlite

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

REVIEWER = "Reviewer"        # your name: it goes in the audit trail next to every decision
RUN_ID = "globex-run-1"      # a new id starts a new run; the same id resumes a paused review
RUN_MODE = "smoke"           # "smoke" = 10 questions, "full" = the whole questionnaire

PIPELINE = "C"               # this notebook uses the Jev hybrid
HAIKU, SONNET = "claude-haiku-5-5", "claude-sonnet-5-5"
DRAFT_MODEL = SONNET
JEV_MODEL = "jev-latest"
PRICES = {SONNET: (2.00, 10.00), HAIKU: (0.10, 0.50), "jev": (0.042, 0.0)}   # USD per million tokens

THRESHOLDS = {"high_min_support": 1.0, "high_min_explicit": 1.0, "medium_min_support": 0.75,
              "min_retrieval_score": 4.0}
JEV_T = {"yes": 0.7, "no": 0.4, "gate_relevant": 0.45, "gate_evidence": 0.55,
         "complete": 0.6, "support_conf": 0.8}            # tuned in run 4
LIBRARY_DB = "answer_library.db"   # approved answers; download it at the end and re-upload next time
CHECKPOINT_DB = "review_checkpoints.db"
LIBRARY_MATCH = 0.8                # Jev probability that a new question asks the same as an approved one
ADD_AUTO_TO_LIBRARY = False        # False: only answers a human approved or edited enter the library

AUTO_APPROVE_LEVELS = {"High"}
ESCALATE_TEXT = "Not covered by policy, escalate"
SEARCH_K = 6
SMOKE_REFS = {"1.1.1", "AI-02", "3.5", "1.5.3", "1.3.2", "1.5.1", "1.5.4", "AI-20", "1.4.2", "3.12"}
print(f"Run '{RUN_ID}', mode '{RUN_MODE}', reviewer '{REVIEWER}'.")

## 2. Load and index the Acme SaaS policy pack
Upload `acme-policy-pack.zip` when asked.

In [ ]:
import glob, pathlib, zipfile

if not glob.glob("**/data/policies/*.md", recursive=True):
    from google.colab import files
    uploaded = files.upload()                      # choose acme-policy-pack.zip
    for name in uploaded:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(".")

POLICY_DIR = pathlib.Path(sorted(glob.glob("**/data/policies", recursive=True))[0])
policy_files = sorted(POLICY_DIR.glob("*.md"))
print(f"{len(policy_files)} policy docs in {POLICY_DIR}:")
for p in policy_files:
    print("  -", p.name)

In [ ]:
import re

def parse_doc(path):
    text = path.read_text(encoding="utf-8")
    meta = {}
    m = re.match(r"^---\n(.*?)\n---\n", text, re.S)          # front matter
    if m:
        for line in m.group(1).splitlines():
            key, _, value = line.partition(":")
            meta[key.strip()] = value.strip()
        text = text[m.end():]
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        head = re.match(r"## ([A-Z]+-\d+) (.+)", section)
        if not head:
            continue
        chunks.append({
            "section_id": head.group(1),
            "heading": head.group(2).strip(),
            "doc_id": meta.get("doc_id", ""),
            "doc_title": meta.get("title", path.stem),
            "version": meta.get("version", ""),
            "text": section[head.end():].strip(),
        })
    return chunks

chunks = [c for p in policy_files for c in parse_doc(p)]
print(f"{len(chunks)} sections. Example:\n")
print(chunks[0]["section_id"], "|", chunks[0]["doc_title"], "|", chunks[0]["heading"])
print(chunks[0]["text"])

In [ ]:
import sqlite3

DB_PATH = "acme_policies.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute('''
    CREATE VIRTUAL TABLE passages USING fts5(
        section_id UNINDEXED, doc_id UNINDEXED, doc_title UNINDEXED, version UNINDEXED,
        heading, text,
        tokenize = 'porter unicode61'
    )''')
conn.executemany(
    "INSERT INTO passages VALUES (:section_id, :doc_id, :doc_title, :version, :heading, :text)",
    chunks)
conn.commit()

STOPWORDS = set('''a an and are as at be by can do does for from has have how i if in
into is it its of on or our please provide the their there this to us we what when
where which who will with you your yours any all describe detail details'''.split())

def to_fts_query(text):
    terms = [t for t in re.findall(r"[a-z0-9]+", text.lower())
             if t not in STOPWORDS and len(t) > 1]
    return " OR ".join(f'"{t}"' for t in dict.fromkeys(terms))   # quoted = safe from FTS syntax

def search(query, k=4):
    fts = to_fts_query(query)
    if not fts:
        return []
    rows = conn.execute('''
        SELECT section_id, doc_title, heading, text,
               -bm25(passages, 0, 0, 0, 0, 2.0, 1.0) AS score
        FROM passages WHERE passages MATCH ?
        ORDER BY score DESC LIMIT ?''', (fts, k)).fetchall()
    return [dict(r) for r in rows]

def format_hits(hits):
    return "\n\n".join(
        f"[{h['section_id']}] {h['doc_title']}: {h['heading']} (score {h['score']:.2f})\n{h['text']}"
        for h in hits)

print(f"Indexed {conn.execute('SELECT count(*) FROM passages').fetchone()[0]} sections.\n")
for q in ["Do you train AI models on customer data?", "Is data encrypted at rest?", "bug bounty programme"]:
    print(f"Q: {q}")
    for h in search(q, k=3):
        print(f"   {h['score']:5.2f}  {h['section_id']:7} {h['heading']}")
    print()

# The batch run calls search from several threads at once, so guard the SQLite connection.
import threading
_search_lock = threading.Lock()
if "_search_unlocked" not in globals():
    _search_unlocked = search
def search(query, k=4):
    with _search_lock:
        return _search_unlocked(query, k)

VERSIONS = {c["section_id"]: c["version"] for c in chunks}          # for spotting stale library answers
TITLES = {c["section_id"]: c["doc_title"] for c in chunks}

## 3. Cost tracking

In [ ]:
import threading
import pandas as pd
from collections import defaultdict

_usage_lock = threading.Lock()
USAGE = defaultdict(lambda: {"calls": 0, "input_tokens": 0, "output_tokens": 0})

def track(model, input_tokens, output_tokens):
    with _usage_lock:
        u = USAGE[model]
        u["calls"] += 1
        u["input_tokens"] += input_tokens or 0
        u["output_tokens"] += output_tokens or 0

def track_msg(model, msg):
    um = getattr(msg, "usage_metadata", None) or {}
    track(model, um.get("input_tokens", 0), um.get("output_tokens", 0))

def cost_table(usage=None):
    rows = []
    for model, u in (usage or USAGE).items():
        p_in, p_out = PRICES.get(model, (0, 0))
        rows.append({"model": model, **u,
                     "cost_usd": u["input_tokens"] / 1e6 * p_in + u["output_tokens"] / 1e6 * p_out})
    df = pd.DataFrame(rows, columns=["model", "calls", "input_tokens", "output_tokens", "cost_usd"])
    return df.sort_values("cost_usd", ascending=False).reset_index(drop=True)

print("Usage tracking ready.")

## 4. Schemas and prompts
The drafting prompt now refers to the prospect as *"the customer"*, and has a separate `reviewer_note` for caveats.

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field

class Classification(BaseModel):
    category: Literal["security", "ai_governance", "privacy"]
    asks_for_commitment: bool = Field(description=(
        "True ONLY if the question asks the vendor to go beyond describing its current practice: "
        "to guarantee or warrant an outcome, agree new or different contractual terms, accept "
        "liability or indemnity, agree SLA penalties or service credits, or hand over a confidential "
        "report. False for questions asking what the vendor's existing policy, timeframe, notice "
        "period or process is, even when the answer describes something the vendor already does."))
    reason: str = Field(description="One short sentence explaining the category.")

class DraftAnswer(BaseModel):
    covered: bool = Field(description="False if the passages do not answer the question.")
    answer: str = Field(description="The answer as the vendor would write it. Empty if not covered.")
    cited_section_ids: list[str] = Field(description="Section IDs the answer relies on, e.g. ['AIG-3'].")
    reviewer_note: str = Field(description=(
        "Internal note for the human reviewer: gaps, caveats or assumptions, e.g. 'policy does not mention "
        "service credits'. Never shown to the customer. '' if there is nothing to flag."))

class Claim(BaseModel):
    claim: str = Field(description="One factual statement taken from the answer.")
    cited_section_id: str = Field(description="The one section that supports it, or '' if none.")
    evidence_quote: str = Field(description="Exact words copied from that section that support the claim, or ''.")
    verdict: Literal["explicit", "inferred", "unsupported"] = Field(description=(
        "explicit: the section states it directly; inferred: it follows from the section but is not "
        "stated; unsupported: no cited section supports it."))

class GroundingReport(BaseModel):
    claims: list[Claim]

class ExtractedClaim(BaseModel):
    claim: str = Field(description="One factual statement taken from the answer.")
    cited_section_id: str = Field(description="The one section that supports it, or '' if none.")
    evidence_quote: str = Field(description="Exact words copied from that section that support the claim, or ''.")

class ClaimList(BaseModel):
    claims: list[ExtractedClaim]

class QState(TypedDict, total=False):
    question: str
    classification: dict
    restricted: bool
    restricted_reasons: list[str]
    caps: list[str]       # reasons that cap confidence at Medium (pipeline C: uncertain Jev answers)
    library: dict         # library lookup result (Phase 3)
    jev: dict             # raw Jev probabilities, logged to the audit trail
    search_log: list[str]
    passages: dict        # section_id -> passage dict
    draft: dict
    grounding: dict
    result: dict

print("Schemas ready.")

In [ ]:
import re
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

LLM = {m: ChatAnthropic(model=m, max_tokens=2048, max_retries=3) for m in (HAIKU, SONNET)}

def structured(model, schema, messages):
    '''Structured call that also records token usage.'''
    out = LLM[model].with_structured_output(schema, method="json_schema", include_raw=True).invoke(messages)
    track_msg(model, out["raw"])
    if out.get("parsing_error"):
        raise out["parsing_error"]
    return out["parsed"]
CLASSIFIER_PROMPT = '''You route questions from a customer security / AI due-diligence questionnaire
to the right specialist at a SaaS vendor.
- security: infosec programme, certifications, access control, encryption, vulnerability management,
  hosting, network, identity/SSO, incident response, business continuity.
- ai_governance: AI features, models, model providers, training on data, AI risk management,
  AI regulation (EU AI Act, NIST AI RMF, ISO 42001), AI testing and oversight.
- privacy: personal data, GDPR, DPAs, data retention and deletion, sub-processors, data transfers,
  data subject rights.'''

COMMON_RULES = '''You work for Acme SaaS (the vendor) and answer questions from a prospect's
due-diligence questionnaire using ONLY Acme's internal policy pack.
Rules:
- Every statement must come from a retrieved policy section. Never use outside knowledge about
  Acme or assume typical industry practice.
- If the sections do not answer the question, say so. Do not guess or fill gaps.
- Do not add guarantees, promises or commitments that the policy does not state.'''

SPECIALIST_FOCUS = {
    "security": "You are Acme's security specialist. Be precise about certifications, scopes, "
                "time frames and technical controls.",
    "ai_governance": "You are Acme's AI governance specialist. Be precise about how customer data "
                     "is (and is not) used by AI, model providers, human oversight and regulatory "
                     "positioning. Never overstate certifications that are only in progress.",
    "privacy": "You are Acme's privacy specialist. Be precise about controller/processor roles, "
               "retention periods, deletion, sub-processors and data locations.",
}

DRAFT_PROMPT = '''Write the questionnaire answer from the passages provided.
- Answer exactly what was asked, in 1 to 3 sentences of plain professional English, in Acme's voice ("Acme ...", "We ...").
- Include the specifics the question asks for (numbers, time frames, standards). Do not add related
  facts the question didn't ask about: every extra sentence is one more claim a reviewer must check.
- For a yes/no question, start with "Yes." or "No." only when a passage states it directly.
- Do not put section IDs in the answer text; list them in cited_section_ids.
- Refer to the prospect as "the customer" (or "customers"), never by their company name, so the answer
  can be reused for other prospects.
- Put anything meant for the internal reviewer (gaps, caveats, "the policy does not say...") in
  reviewer_note, never in the answer.
- If the passages do not answer the question, set covered=false, answer="" and cited_section_ids=[].'''

GROUNDING_PROMPT = '''You are an auditor checking a questionnaire answer against its cited policy sections.
Split the answer into its individual factual claims. A bare "Yes." or "No." is not a separate claim:
treat it as part of the claim that follows. For each claim, pick the one cited section that supports
it and copy the exact supporting words from that section into evidence_quote (verbatim, no
paraphrasing). Verdicts:
- explicit: the quoted words state the same fact, even if the answer words it differently;
- inferred: the claim adds meaning the quote does not contain (a conclusion, an absence, a wider scope);
- unsupported: no cited section supports it.
Be strict about meaning, not wording: anything not in the cited sections is unsupported.'''
print("Models and prompts ready.")

EXTRACT_PROMPT = '''Split the questionnaire answer into its individual factual claims. A bare "Yes." or
"No." is not a separate claim: treat it as part of the claim that follows. For each claim, pick the one
cited section that supports it and copy the exact supporting words from that section into evidence_quote:
verbatim, no paraphrasing, no ellipses. If no cited section supports a claim, leave both fields empty.'''

RESTRICTED_RULES = {
    "guarantee":           r"\bguarantee",
    "warranty":            r"\bwarrant",
    "liability":           r"\bliabilit|\bliable\b",
    "indemnity":           r"\bindemn",
    "commitment":          r"\bcommit(ment)? to\b|\bcontractual|\bin (the|our|your) contract\b",
    "sla / credits":       r"\bSLA\b|service credit|\bpenalt",
    "confidential report": r"\b(full|complete) (pen(etration)? test|audit|soc ?2) report",
}
def keyword_flags(q):
    return [f"keyword rule: {name}" for name, rx in RESTRICTED_RULES.items() if re.search(rx, q, re.I)]

def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s%.]", " ", s.lower())).strip()

NUM = re.compile(r"\d+(?:\.\d+)?")
def numbers_match(claim, section_text):
    '''Every number in the claim must appear in the cited section (Jev is weak at numbers).'''
    have = set(NUM.findall(section_text))
    return all(n in have for n in NUM.findall(claim))
print("Prompts and checks ready.")

In [ ]:
def score(state: QState) -> dict:
    d, claims = state["draft"], state["grounding"]["claims"]
    n = len(claims)
    support = sum(c["verdict"] != "unsupported" for c in claims) / n if n else 0.0
    explicit = sum(c["verdict"] == "explicit" for c in claims) / n if n else 0.0
    best = max((state["passages"][s]["score"] for s in d["cited_section_ids"] if s in state["passages"]), default=0.0)
    caps = state.get("caps", [])
    signals = {"library_match": False, "best_retrieval_score": round(best, 2), "claims": n,
               "claim_support": round(support, 2), "explicit_evidence": round(explicit, 2),
               "restricted": state["restricted"], **state.get("jev", {})}
    reasons, answer = [], d["answer"]

    if not d["covered"] or n == 0 or support == 0:
        level, answer = "Low", ESCALATE_TEXT
        reasons.append("policy pack doesn't cover this question")
    elif (support >= THRESHOLDS["high_min_support"] and explicit >= THRESHOLDS["high_min_explicit"]
          and (PIPELINE == "C" or best >= THRESHOLDS["min_retrieval_score"]) and not caps):
        level = "High"
        reasons.append("every claim is backed by a verified quote from a cited section")
    elif support >= THRESHOLDS["medium_min_support"]:
        level = "Medium"
        if explicit < THRESHOLDS["high_min_explicit"]:
            reasons.append("some claims are inferred rather than stated")
        if support < 1:
            reasons.append("some claims are unsupported")
        reasons += caps
    else:
        level = "Low"
        reasons.append(f"only {support:.0%} of claims are supported")
    if any(c.get("contradicted") for c in claims):
        level = "Low"
        reasons.append("a claim contradicts the cited policy")
    if state["restricted"]:
        level = "Low"
        reasons += [f"restricted topic, {r}" for r in state["restricted_reasons"]]

    status = ("escalate" if answer == ESCALATE_TEXT
              else "auto_approve" if level in AUTO_APPROVE_LEVELS else "human_review")
    return {"result": {"question": state["question"], "category": state["classification"]["category"],
                       "answer": answer, "draft_answer": d["answer"],
                       "sources": d["cited_section_ids"] if answer != ESCALATE_TEXT else [],
                       "confidence": level, "status": status, "reasons": reasons, "signals": signals,
                       "reviewer_note": d.get("reviewer_note", "")}}
print("Scoring ready.")

## 5. The Jev pipeline (unchanged from run 4)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from typesafe_sdk import TypeSafeClient, Noul, Choice

jev_client = TypeSafeClient(model=JEV_MODEL)          # reads TYPESAFE_API_KEY; retries 429s/timeouts itself
_jev_pool = ThreadPoolExecutor(max_workers=8)

def ask_jev(state, questions):
    r = jev_client.system_one(state=state, questions=questions, model=JEV_MODEL)
    track("jev", r.usage.input_tokens, r.usage.output_tokens)
    return r.answers

CLASSIFY_QS = {
    "category": Choice(
        instructions="Which specialist at a SaaS vendor should answer this due-diligence question?",
        criteria={
            "security": "Information security programme, certifications, access control, encryption, "
                        "vulnerabilities, hosting, network, identity/SSO, incident response, continuity.",
            "ai_governance": "AI features, models and model providers, training on data, AI risk "
                             "management, AI regulation (EU AI Act, NIST AI RMF, ISO 42001), AI testing.",
            "privacy": "Personal data, GDPR, data processing agreements, retention and deletion, "
                       "sub-processors, data transfers, data subject rights."}),
    "asks_guarantee": Noul(instructions="The question asks the vendor to guarantee or warrant an outcome."),
    "asks_contract_terms": Noul(instructions=(
        "The question asks the vendor to agree new contractual terms, an SLA, penalties or service "
        "credits, rather than to describe its existing policy or practice.")),
    "asks_liability": Noul(instructions="The question asks the vendor to accept liability or to indemnify the customer."),
    "asks_confidential_document": Noul(instructions=(
        "The question asks the vendor to hand over a complete confidential document, such as a full "
        "audit report or penetration test report.")),
}
COMMIT_KEYS = ["asks_guarantee", "asks_contract_terms", "asks_liability", "asks_confidential_document"]

def classify_c(state: QState) -> dict:
    q = state["question"]
    a = ask_jev({"questionnaire_question": q}, CLASSIFY_QS)
    p = {k: round(a[k].noul, 3) for k in COMMIT_KEYS}
    reasons = keyword_flags(q) + [f"Jev: {k} p={v:.2f}" for k, v in p.items() if v >= JEV_T["yes"]]
    caps = [f"Jev unsure whether it {k.replace('_', ' ')} (p={v:.2f})"
            for k, v in p.items() if JEV_T["no"] <= v < JEV_T["yes"]]
    cat = a["category"]
    return {"classification": {"category": cat.choice, "asks_for_commitment": bool(reasons),
                               "reason": f"Jev confidence {cat.confidence:.2f}"},
            "restricted": bool(reasons), "restricted_reasons": reasons, "caps": caps,
            "jev": {"category_conf": round(cat.confidence, 3), **{f"p_{k}": v for k, v in p.items()}}}

GATE_QS = {
    "is_relevant": Noul(instructions="The passage addresses the subject of the question."),
    "has_evidence": Noul(instructions="The passage states information that directly answers the question, or at least one part of it."),
}
COMPLETE_Q = {"answers_every_part": Noul(instructions=(
    "Taken together, the passages contain the information needed to answer every part of the question."))}

def retrieve_gate_c(state: QState) -> dict:
    q = state["question"]
    hits = search(q, k=SEARCH_K)
    def judge(h):
        a = ask_jev({"question": q, "passage": {"id": h["section_id"], "title": h["heading"], "text": h["text"]}}, GATE_QS)
        return h, a["is_relevant"].noul, a["has_evidence"].noul
    judged = list(_jev_pool.map(judge, hits))
    kept = {h["section_id"]: {**h, "p_relevant": round(r, 3), "p_evidence": round(e, 3)}
            for h, r, e in judged if r >= JEV_T["gate_relevant"] and e >= JEV_T["gate_evidence"]}
    jev = {**state.get("jev", {}), "gate": {h["section_id"]: [round(r, 2), round(e, 2)] for h, r, e in judged},
           "sections_kept": len(kept)}
    caps = list(state.get("caps", []))
    if kept:
        a = ask_jev({"question": q, "passages": [{"id": s, "text": h["text"]} for s, h in kept.items()]}, COMPLETE_Q)
        jev["p_complete"] = round(a["answers_every_part"].noul, 3)
        if jev["p_complete"] < JEV_T["complete"]:
            caps.append(f"sections may not answer every part of the question (p={jev['p_complete']:.2f})")
    return {"passages": kept, "jev": jev, "caps": caps,
            "search_log": [f"(question) {q}: {len(hits)} found, {len(kept)} passed the Jev gate"]}

def draft_c(state: QState) -> dict:
    q, category = state["question"], state["classification"]["category"]
    if not state["passages"]:
        return {"draft": {"covered": False, "answer": "", "cited_section_ids": []}}
    ranked = sorted(state["passages"].values(), key=lambda h: -h["score"])
    d = structured(DRAFT_MODEL, DraftAnswer, [
        SystemMessage(COMMON_RULES + "\n\n" + SPECIALIST_FOCUS[category] + "\n\n" + DRAFT_PROMPT),
        HumanMessage(f"Question: {q}\n\nPolicy sections:\n\n{format_hits(ranked)}")]).model_dump()
    d["cited_section_ids"] = [s for s in dict.fromkeys(d["cited_section_ids"]) if s in state["passages"]]
    return {"draft": d}

RELATION_Q = {"relation": Choice(
    instructions="How does the section relate to the claim?",
    criteria={"supports": "The section states the claim or directly implies that it is true.",
              "contradicts": "The section states the opposite of the claim or implies that it is false.",
              "says_nothing": "The section does not address the claim either way."})}

def verify_c(state: QState) -> dict:
    d = state["draft"]
    if not d["covered"] or not d["cited_section_ids"]:
        return {"grounding": {"claims": []}}
    cited = {s: state["passages"][s] for s in d["cited_section_ids"]}
    extracted = structured(HAIKU, ClaimList, [
        SystemMessage(EXTRACT_PROMPT),
        HumanMessage(f"Answer: {d['answer']}\n\nCited sections:\n\n{format_hits(list(cited.values()))}")])

    def check(c):
        c = c.model_dump()
        section = cited.get(c["cited_section_id"])
        if section is None and len(cited) == 1:            # Haiku left the source blank: only one candidate
            c["cited_section_id"], section = next(iter(cited.items()))
        quote = c["evidence_quote"].strip()
        c["quote_found"] = bool(section and quote and _norm(quote) in _norm(section["text"]))
        c["numbers_ok"] = bool(section) and numbers_match(c["claim"], section["text"])
        if section is None:
            c["verdict"], c["note"] = "unsupported", "cited a section that wasn't retrieved"
            return c
        if quote and not c["quote_found"]:
            c["verdict"], c["note"] = "unsupported", "quote not found in the section"
            return c
        rel = ask_jev({"claim": c["claim"], "section": section["text"]}, RELATION_Q)["relation"]
        c["jev_relation"], c["jev_confidence"] = rel.choice, round(rel.confidence, 3)
        if rel.choice == "contradicts":
            c["verdict"], c["contradicted"], c["note"] = "unsupported", True, f"Jev: contradicts ({rel.confidence:.2f})"
        elif rel.choice == "says_nothing":
            c["verdict"], c["note"] = "unsupported", f"Jev: section says nothing about it ({rel.confidence:.2f})"
        elif not c["numbers_ok"]:
            c["verdict"], c["note"] = "unsupported", "a number in the claim isn't in the section"
        elif not quote:
            c["verdict"], c["note"] = "inferred", "no quote given; Jev judged the whole section"
        elif rel.confidence >= JEV_T["support_conf"]:
            c["verdict"] = "explicit"
        else:
            c["verdict"], c["note"] = "inferred", f"Jev: supports, low confidence ({rel.confidence:.2f})"
        return c

    return {"grounding": {"claims": list(_jev_pool.map(check, extracted.claims))}}

print("Pipeline C nodes defined.")

## 6. The answer library
A small SQLite database with a full-text index over the approved questions. A lookup takes three steps:
1. **Search** for the 3 most similar approved questions.
2. **Drop stale answers**: any whose policy sections have a different version from when the answer was approved.
3. **Ask Jev**, for each remaining candidate: *"Do both questions ask for the same information, so the approved answer would fully answer the new one?"* The best match above `LIBRARY_MATCH` is reused.

If you have an `answer_library.db` from an earlier session, upload it here. Otherwise press Cancel to start with an empty library.

In [ ]:
import json, datetime
from google.colab import files

if not os.path.exists(LIBRARY_DB):
    print("Optional: upload answer_library.db from an earlier session, or press Cancel to start empty.")
    try:
        files.upload()
    except Exception:
        pass

lib = sqlite3.connect(LIBRARY_DB, check_same_thread=False)
lib.row_factory = sqlite3.Row
lib.executescript('''
    CREATE TABLE IF NOT EXISTS answers (
        id INTEGER PRIMARY KEY, question TEXT, answer TEXT, sources TEXT, category TEXT,
        policy_versions TEXT, decision TEXT, reviewer TEXT, approved_at TEXT, run_id TEXT,
        times_reused INTEGER DEFAULT 0);
    CREATE VIRTUAL TABLE IF NOT EXISTS answers_fts USING fts5(question, tokenize = 'porter unicode61');''')
_lib_lock = threading.Lock()

SAME_Q = {"same": Noul(instructions=(
    "Both questions ask for the same information, so an answer to the approved question would fully "
    "answer the new question."))}

def library_lookup(question):
    fts = to_fts_query(question)
    if not fts:
        return None
    with _lib_lock:
        rows = [dict(r) for r in lib.execute('''
            SELECT a.*, -bm25(answers_fts) AS score FROM answers_fts JOIN answers a ON a.id = answers_fts.rowid
            WHERE answers_fts MATCH ? ORDER BY score DESC LIMIT 3''', (fts,)).fetchall()]
    best, checked = None, []
    for row in rows:
        stored = json.loads(row["policy_versions"] or "{}")
        stale = [s for s, v in stored.items() if VERSIONS.get(s) != v]
        if stale:
            checked.append({"id": row["id"], "stale": stale})
            continue
        p = ask_jev({"new_question": question, "approved_question": row["question"]}, SAME_Q)["same"].noul
        checked.append({"id": row["id"], "p_same": round(p, 3)})
        if p >= LIBRARY_MATCH and (best is None or p > best["p_same"]):
            best = {**row, "p_same": round(p, 3)}
    return {"match": best, "checked": checked}

def library_add(question, answer, sources, category, decision, reviewer):
    versions = {s: VERSIONS.get(s) for s in sources}
    with _lib_lock:
        cur = lib.execute('''INSERT INTO answers (question, answer, sources, category, policy_versions, decision,
                             reviewer, approved_at, run_id) VALUES (?,?,?,?,?,?,?,?,?)''',
                          (question, answer, json.dumps(sources), category, json.dumps(versions), decision,
                           reviewer, datetime.datetime.now().isoformat(timespec="seconds"), RUN_ID))
        lib.execute("INSERT INTO answers_fts (rowid, question) VALUES (?, ?)", (cur.lastrowid, question))
        lib.commit()

def library_size():
    with _lib_lock:
        return lib.execute("SELECT count(*) FROM answers").fetchone()[0]

print(f"Library ready: {library_size()} approved answers.")

## 7. The per-question graph, now checking the library first
After the Jev classification, a library hit skips the whole search, draft and check chain.
- **Not restricted:** the approved answer is reused and auto-approved.
- **Restricted** (a guarantee, contract terms, liability and so on): the approved answer is offered to the reviewer as a draft, because commitments always get a fresh human look.

In [ ]:
from langgraph.graph import StateGraph, START, END

def library_node(state: QState) -> dict:
    if library_size() == 0:
        return {"library": None}
    return {"library": library_lookup(state["question"])}

def library_answer(state: QState) -> dict:
    m = state["library"]["match"]
    reason = (f"reused approved answer #{m['id']} ({m['decision']} by {m['reviewer']} on {m['approved_at'][:10]}; "
              f"Jev same-question p={m['p_same']:.2f})")
    if state["restricted"]:
        level, status = "Medium", "human_review"
        reasons = [reason, *[f"restricted topic, {r}" for r in state["restricted_reasons"]],
                   "commitments are always re-checked by a human"]
    else:
        level, status, reasons = "High", "auto_approve", [reason]
    return {"result": {"question": state["question"], "category": state["classification"]["category"],
                       "answer": m["answer"], "draft_answer": m["answer"], "sources": json.loads(m["sources"]),
                       "confidence": level, "status": status, "reasons": reasons, "reviewer_note": "",
                       "signals": {"library_match": True, "library_id": m["id"], "p_same": m["p_same"],
                                   "restricted": state["restricted"], **state.get("jev", {})}},
            "passages": {}, "grounding": {"claims": []}, "search_log": ["(library hit)"]}

def after_library(state: QState):
    return "library_answer" if state.get("library") and state["library"]["match"] else "search_and_jev_gate"

b = StateGraph(QState)
for name, fn in [("jev_classify", classify_c), ("library_lookup", library_node), ("library_answer", library_answer),
                 ("search_and_jev_gate", retrieve_gate_c), ("draft", draft_c), ("verify", verify_c), ("score", score)]:
    b.add_node(name, fn)
b.add_edge(START, "jev_classify")
b.add_edge("jev_classify", "library_lookup")
b.add_conditional_edges("library_lookup", after_library, ["library_answer", "search_and_jev_gate"])
b.add_edge("library_answer", END)
b.add_edge("search_and_jev_gate", "draft"); b.add_edge("draft", "verify"); b.add_edge("verify", "score")
b.add_edge("score", END)
graph = b.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 8. Load the questionnaire
Upload `globex_vendor_due_diligence_2026.xlsx` (and `golden_set.csv` if you like). Hold Ctrl or Cmd to select both.

In [ ]:
from google.colab import files

print("Choose the questionnaire (XLSX or CSV), plus golden_set.csv if you have it.")
try:
    uploaded = files.upload()
except Exception:
    uploaded = {}
candidates = [n for n in uploaded if not n.startswith("golden_set")]
if candidates:
    QFILE = pathlib.Path(candidates[0])
else:
    QFILE = pathlib.Path(sorted(glob.glob("**/data/questionnaires/*.xlsx", recursive=True))[0])
print("Questionnaire:", QFILE)

In [ ]:
import csv, openpyxl

Q_HDR   = re.compile(r"^\s*(question|questions|question text|requirement|control question)\b", re.I)
REF_HDR = re.compile(r"^\s*(ref\.?|reference|q\s*#|q\s*no\.?|#|id|no\.?|item|question id|control id)\s*$", re.I)
ANS_HDRS = [r"supplier response", r"vendor response", r"\banswer\b", r"supporting comments",
            r"\bcomments?\b", r"\bresponse\b"]                    # first match wins
CHOICE_HDR = re.compile(r"^\s*(response|yes\s*/\s*no|y\s*/\s*n|compliant\??)\s*$", re.I)
EVID_HDR = re.compile(r"evidence|supporting doc", re.I)
NOT_OURS = re.compile(r"reviewer|internal|office use|assessor|score|weight", re.I)   # columns we must not fill
SUBQ = re.compile(r"^\s*\(?([a-h]|i{1,3}|iv|v)\)\s+")

def _txt(v):
    return "" if v is None else str(v).strip()

def read_grid(path):
    # -> {sheet: {"rows": {r: [values]}, "merged": {r: text}}}; merged = rows merged across columns
    if path.suffix.lower() == ".csv":
        with open(path, newline="", encoding="utf-8-sig") as f:
            rows = list(csv.reader(f))
        return {"csv": {"rows": {i + 1: r for i, r in enumerate(rows)}, "merged": {}}}
    wb = openpyxl.load_workbook(path)
    out = {}
    for ws in wb.worksheets:
        if ws.sheet_state != "visible":
            continue
        rows = {r: [c.value for c in row] for r, row in enumerate(ws.iter_rows(), start=1)}
        merged = {}
        for rng in ws.merged_cells.ranges:
            if rng.max_col > rng.min_col:
                for r in range(rng.min_row, rng.max_row + 1):
                    merged[r] = _txt(ws.cell(rng.min_row, rng.min_col).value)
        out[ws.title] = {"rows": rows, "merged": merged}
    return out

def find_layout(rows, scan=25):
    for r in sorted(rows)[:scan]:
        hdr = {c: _txt(v) for c, v in enumerate(rows[r]) if _txt(v)}
        q_col = next((c for c, h in hdr.items() if Q_HDR.search(h) and not NOT_OURS.search(h)), None)
        if q_col is None:
            continue
        ans_col = None
        for pat in ANS_HDRS:
            ans_col = next((c for c, h in hdr.items() if c != q_col and re.search(pat, h, re.I)
                            and not NOT_OURS.search(h)), None)
            if ans_col is not None:
                break
        if ans_col is None:
            continue
        ref_col = next((c for c, h in hdr.items() if REF_HDR.match(h)), None)
        choice_col = next((c for c, h in hdr.items() if c not in (ans_col, q_col) and CHOICE_HDR.match(h)), None)
        evid_col = next((c for c, h in hdr.items() if c != ans_col and EVID_HDR.search(h)
                         and not NOT_OURS.search(h)), None)
        return {"header_row": r, "q": q_col, "ref": ref_col, "answer": ans_col,
                "choice": choice_col, "evidence": evid_col, "headers": hdr}
    return None

def extract_questions(path):
    items, report = [], []
    for sheet, g in read_grid(path).items():
        rows, merged = g["rows"], g["merged"]
        lay = find_layout(rows)
        if not lay:
            report.append(f"  {sheet}: no question table found, skipped")
            continue
        h = lay["headers"]
        report.append(f"  {sheet}: header row {lay['header_row']} | question='{h[lay['q']]}' "
                      f"answer='{h[lay['answer']]}'"
                      + (f" ref='{h[lay['ref']]}'" if lay["ref"] is not None else "")
                      + (f" evidence='{h[lay['evidence']]}'" if lay["evidence"] is not None else ""))
        section, n, prefilled = "", 0, 0
        for r in sorted(rows):
            if r <= lay["header_row"]:
                continue
            vals = list(rows[r]) + [None] * 30
            if r in merged:                                   # section header / footer
                section = merged[r]
                continue
            q = _txt(vals[lay["q"]])
            if len(q) < 10 or not re.search(r"[A-Za-z]{3}", q):
                continue
            if _txt(vals[lay["answer"]]):
                prefilled += 1
                continue
            ref = _txt(vals[lay["ref"]]) if lay["ref"] is not None else ""
            model_q = re.sub(r"\s+", " ", q)
            if SUBQ.match(q) and section:                     # a) b) c) -> add parent context
                parent = re.sub(r"^[\d.]+\s*", "", section)
                model_q = f"{parent}: {SUBQ.sub('', model_q)}"
            items.append({"sheet": sheet, "row": r, "ref": ref or f"{sheet}!{r}", "question": q,
                          "model_question": model_q, "section": section,
                          "answer_col": lay["answer"], "evidence_col": lay["evidence"]})
            n += 1
        report[-1] += f"\n      -> {n} questions" + (f", {prefilled} already answered (skipped)" if prefilled else "")
    return items, report

items, report = extract_questions(QFILE)
print(f"{QFILE.name}:")
print("\n".join(report))
print(f"\n{len(items)} questions to answer. First few:")
for it in items[:3] + [i for i in items if i["model_question"] != re.sub(r'\s+', ' ', i["question"])][:2]:
    print(f"  [{it['sheet']} {it['ref']}] {it['model_question'][:110]}")

if RUN_MODE == "smoke":
    items = [i for i in items if i["ref"] in SMOKE_REFS]
print(f"\nRun mode '{RUN_MODE}': {len(items)} questions will be answered by pipeline {PIPELINE}.")

if RUN_MODE == "smoke":
    items = [i for i in items if i["ref"] in SMOKE_REFS]
print(f"\nRun mode '{RUN_MODE}': {len(items)} questions.")

In [ ]:
# ---- Pre-flight: free apart from three tiny API pings ----
checks = []
def check(name, ok, hint=""):
    checks.append(ok)
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  ->  {hint}"))

check("Reviewer name set", REVIEWER.strip() not in ("", "Reviewer"), "put your name in REVIEWER in section 1 (it goes in the audit trail)")
check("Drafting prompt has the Phase 3 changes", "never by their company name" in DRAFT_PROMPT, "re-run section 4")
check("Policy index loaded (67 sections)", conn.execute("SELECT count(*) FROM passages").fetchone()[0] == 67, "re-run section 2")
expected = len(SMOKE_REFS) if RUN_MODE == "smoke" else 60
check(f"Questions selected: {len(items)} (expected {expected} for the Globex file)", len(items) == expected,
      "check the upload above; for another questionnaire this can be ignored")
check(f"Library open ({library_size()} approved answers)", True)
for model in (HAIKU, SONNET):
    try:
        ChatAnthropic(model=model, max_tokens=5).invoke("Reply with OK")
        check(f"Anthropic key and credits work for {model}", True)
    except Exception as e:
        check(f"Anthropic key and credits work for {model}", False, f"{type(e).__name__}: {str(e)[:150]}")
try:
    a = ask_jev("The sky is blue.", {"ping": Noul(instructions="The statement is about the sky.")})
    check(f"TypeSafe key works (Jev ping p={a['ping'].noul:.2f})", True)
except Exception as e:
    check("TypeSafe key works", False, f"{type(e).__name__}: {str(e)[:150]}")
print(f"\nRun '{RUN_ID}': estimated cost about ${0.003 * len(items):.2f}, less for library hits.")
print("🟢 All clear: run section 9." if all(checks) else "🔴 Fix the ❌ items above before running section 9.")

## 9. The review graph: answer everything, then pause for a human
One graph does the whole job, and a **SQLite checkpointer** saves its state after every step:

```
START ──Send──▶ answer_one × N (in parallel) ──▶ review (one interrupt per question) ──▶ finalise ──▶ END
```

- `answer_one` runs the per-question graph from section 7.
- `review` walks the queue: restricted questions first, then escalations, Low and Medium. It calls `interrupt()` for each one, which pauses the graph and hands the item to you in section 10.
- `finalise` applies your decisions, adds approved and edited answers to the library, and counts library reuses.

Each time you resume, LangGraph re-runs `review` from the top and replays your earlier decisions in order. That's why `review` itself only *collects* decisions, and every write happens in `finalise`, once.

In [ ]:
import operator, time
from typing import Annotated
from langgraph.types import Send, interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver

class RunState(TypedDict):
    items: list
    results: Annotated[list, operator.add]
    decisions: list
    final: list
    summary: dict

def fan_out(state: RunState):
    return [Send("answer_one", item) for item in state["items"]]

def answer_one(item: dict) -> dict:
    t0 = time.time()
    try:
        s = graph.invoke({"question": item["model_question"]})
        r = {**s["result"], "claims": s["grounding"]["claims"], "search_log": s["search_log"]}
    except Exception as e:
        r = {"question": item["model_question"], "answer": "", "draft_answer": "", "sources": [], "category": "",
             "confidence": "Low", "status": "error", "reasons": [f"{type(e).__name__}: {e}"], "reviewer_note": "",
             "signals": {}, "claims": [], "search_log": []}
    r["source_titles"] = [f"{TITLES.get(s, '?')} §{s}" for s in r["sources"]]
    r["seconds"] = round(time.time() - t0, 1)
    return {"results": [{**r, **item}]}

PRIORITY = {"restricted": 0, "escalate": 1, "error": 2, "Low": 3, "Medium": 4}
def review_queue(results):
    todo = [r for r in results if r["status"] != "auto_approve"]
    key = lambda r: (PRIORITY["restricted"] if r["signals"].get("restricted") else
                     PRIORITY.get(r["status"], PRIORITY.get(r["confidence"], 5)), r["sheet"], r["row"])
    return sorted(todo, key=key)

def review(state: RunState) -> dict:
    queue, decisions = review_queue(state["results"]), []
    for n, r in enumerate(queue, start=1):
        payload = {"n": n, "total": len(queue), "key": [r["sheet"], r["row"]], "ref": r["ref"], "sheet": r["sheet"],
                   "question": r["question"], "status": r["status"], "confidence": r["confidence"],
                   "draft": r["draft_answer"], "reviewer_note": r.get("reviewer_note", ""),
                   "sources": r["source_titles"], "reasons": r["reasons"],
                   "claims": [f"{c['verdict']}: {c['claim']}" + (f" ({c['note']})" if c.get("note") else "")
                              for c in r.get("claims", [])]}
        decisions.append({"key": [r["sheet"], r["row"]], **interrupt(payload)})
    return {"decisions": decisions}

def finalise(state: RunState) -> dict:
    by_key = {tuple(d["key"]): d for d in state["decisions"]}
    final, added, reused = [], 0, 0
    for r in state["results"]:
        d = by_key.get((r["sheet"], r["row"]))
        from_library = bool(r["signals"].get("library_match"))
        if d is None:                                          # auto-approved, no human needed
            f = {**r, "final_answer": r["answer"], "decision": "auto_approved", "reviewer": "", "decided_at": ""}
            if ADD_AUTO_TO_LIBRARY and not from_library:
                library_add(r["model_question"], r["answer"], r["sources"], r["category"], "auto_approved", "pipeline (High)")
                added += 1
        elif d["action"] in ("approve", "edit"):
            text = r["draft_answer"] if d["action"] == "approve" else d["answer"]
            f = {**r, "final_answer": text, "decision": "approved" if d["action"] == "approve" else "edited",
                 "reviewer": d["reviewer"], "decided_at": d["at"], "review_comment": d.get("comment", "")}
            if not from_library:                               # already in the library
                library_add(r["model_question"], text, r["sources"], r["category"], f["decision"], d["reviewer"])
                added += 1
        else:
            f = {**r, "final_answer": "", "decision": "rejected", "reviewer": d["reviewer"], "decided_at": d["at"],
                 "review_comment": d.get("comment", "")}
        if from_library:
            reused += 1
            with _lib_lock:
                lib.execute("UPDATE answers SET times_reused = times_reused + 1 WHERE id = ?", (r["signals"]["library_id"],))
                lib.commit()
        final.append(f)
    return {"final": final, "summary": {"added_to_library": added, "library_reuses": reused}}

rb = StateGraph(RunState)
rb.add_node("answer_one", answer_one); rb.add_node("review", review); rb.add_node("finalise", finalise)
rb.add_conditional_edges(START, fan_out, ["answer_one"])
rb.add_edge("answer_one", "review"); rb.add_edge("review", "finalise"); rb.add_edge("finalise", END)
_ck = sqlite3.connect(CHECKPOINT_DB, check_same_thread=False)
run_graph = rb.compile(checkpointer=SqliteSaver(_ck))
config = {"configurable": {"thread_id": RUN_ID}, "max_concurrency": 6}

if run_graph.get_state(config).values:
    print(f"Run '{RUN_ID}' already exists: carry on reviewing in section 10, or set a new RUN_ID in section 1.")
else:
    USAGE.clear()
    t0, done = time.time(), 0
    print(f"Answering {len(items)} questions (library: {library_size()} approved answers)...\n")
    for update in run_graph.stream({"items": items, "results": [], "decisions": [], "final": [], "summary": {}},
                                   config, stream_mode="updates"):
        for r in (update.get("answer_one") or {}).get("results", []):
            done += 1
            icon = "📚" if r["signals"].get("library_match") else {"auto_approve": "🟢", "human_review": "🟡",
                                                                    "escalate": "🔴", "error": "⚠️"}[r["status"]]
            print(f"{done:3}/{len(items)} {icon} {r['status']:12} [{r['sheet'][:18]:18} {r['ref']:7}] {r['question'][:60]}")
    RUN_SECONDS = round(time.time() - t0)
    res = run_graph.get_state(config).values["results"]
    print(f"\nDone in {RUN_SECONDS}s, ${cost_table()['cost_usd'].sum():.4f}. "
          f"{sum(r['status'] == 'auto_approve' for r in res)} auto-approved "
          f"({sum(r['status'] == 'auto_approve' and bool(r['signals'].get('library_match')) for r in res)} reused from the library), "
          f"{len(review_queue(res))} waiting for review in section 10.")

## 10. Review
For each question you'll see the draft, the reviewer note, the sources and why it needs a human. Type:
- **`a`** to approve the draft as it is;
- **`e`** to edit, i.e. type the answer you want to send;
- **`r`** to reject, leaving the answer blank and marked for a subject-matter expert;
- **`q`** to stop for now. Re-run this cell any time to continue where you left off.

Approved and edited answers go into the library when the review finishes.

In [ ]:
def show_item(p):
    print("=" * 100)
    print(f"Review {p['n']} of {p['total']}   [{p['sheet']} {p['ref']}]   {p['status']} / {p['confidence']}")
    print(f"\nQ: {p['question']}\n")
    print(f"Draft: {p['draft'] or '(no draft: the policy pack does not cover this)'}")
    if p["reviewer_note"]:
        print(f"Note for reviewer: {p['reviewer_note']}")
    print(f"Sources: {', '.join(p['sources']) or 'none'}")
    print("Why it needs you:")
    for why in p["reasons"]:
        print(f"   - {why}")
    for c in p["claims"]:
        print(f"   · {c}")

def get_decision(p):
    while True:
        choice = input("\n[a]pprove  [e]dit  [r]eject  [q]uit for now > ").strip().lower()
        if choice == "q":
            return None
        if choice == "a" and not p["draft"]:
            print("There's no draft to approve: choose [e] to write one, or [r].")
            continue
        if choice in ("a", "e", "r"):
            answer = input("Answer to send: ").strip() if choice == "e" else ""
            comment = input("Comment for the audit trail (optional, press Enter to skip): ").strip()
            return {"action": {"a": "approve", "e": "edit", "r": "reject"}[choice], "answer": answer,
                    "comment": comment, "reviewer": REVIEWER,
                    "at": datetime.datetime.now().isoformat(timespec="seconds")}

while True:
    pending = run_graph.get_state(config).interrupts
    if not pending:
        break
    p = pending[0].value
    show_item(p)
    decision = get_decision(p)
    if decision is None:
        print(f"\nPaused at {p['n']} of {p['total']}. Re-run this cell to continue; nothing is lost.")
        break
    run_graph.invoke(Command(resume=decision), config)

state = run_graph.get_state(config)
if not state.next:
    print(f"\n✅ Review complete. {state.values['summary']['added_to_library']} answers added to the library "
          f"(now {library_size()}); {state.values['summary']['library_reuses']} answers reused from it.")

## 11. Save the completed questionnaire, audit trail and library
- **The questionnaire** gets the *final* answers written into the prospect's own cells: green for auto-approved, approved or edited answers, red for rejected ones marked `[NEEDS SME]`.
- **The audit CSV** records, for every question, the draft and the final answer, the decision, the reviewer and the time, plus the full evidence trail.
- **`answer_library.db`** holds the library. Keep it, and upload it in section 6 next time.

In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment

final = run_graph.get_state(config).values.get("final")
assert final, "Finish the review in section 10 first."
GREEN, RED = "E2EFDA", "F8CBAD"
stem = f"{QFILE.stem}_{RUN_ID}"
OUT_XLSX, OUT_AUDIT, OUT_LIB = f"{stem}_FINAL.xlsx", f"{stem}_audit.csv", f"answer_library_{RUN_ID}.csv"

wb = openpyxl.load_workbook(QFILE)
for r in final:
    ws = wb[r["sheet"]]
    text = r["final_answer"] or "[NEEDS SME]"
    c = ws.cell(r["row"], r["answer_col"] + 1, text)
    c.fill = PatternFill("solid", fgColor=RED if r["decision"] == "rejected" else GREEN)
    c.alignment = Alignment(wrap_text=True, vertical="top")
    if r["evidence_col"] is not None and r["source_titles"] and r["final_answer"]:
        ws.cell(r["row"], r["evidence_col"] + 1, "; ".join(r["source_titles"]))
wb.save(OUT_XLSX)

audit = pd.DataFrame([{
    "sheet": r["sheet"], "ref": r["ref"], "row": r["row"], "question": r["question"],
    "specialist": r["category"], "confidence": r["confidence"], "status": r["status"],
    "library_hit": bool(r["signals"].get("library_match")), "draft_answer": r["draft_answer"],
    "reviewer_note": r.get("reviewer_note", ""), "final_answer": r["final_answer"], "decision": r["decision"],
    "edited": r["decision"] == "edited", "reviewer": r["reviewer"], "decided_at": r["decided_at"],
    "review_comment": r.get("review_comment", ""), "sources": "; ".join(r["sources"]),
    "reasons": " | ".join(r["reasons"]), "signals": json.dumps(r["signals"]), "claims": json.dumps(r["claims"]),
    "run_id": RUN_ID} for r in final])
audit.to_csv(OUT_AUDIT, index=False)
pd.read_sql("SELECT * FROM answers", lib).to_csv(OUT_LIB, index=False)

print(audit["decision"].value_counts().to_string(), "\n")
print(f"Library: {library_size()} approved answers. Saved {OUT_XLSX}, {OUT_AUDIT}, {OUT_LIB} and {LIBRARY_DB}.")
for f in (OUT_XLSX, OUT_AUDIT, OUT_LIB, LIBRARY_DB):
    files.download(f)

## 12. See the library at work
Set a new `RUN_ID` in section 1, e.g. `"globex-run-2"`, and run sections **1, 9 and 10** again. This time every question you approved comes back as 📚, reused from the library without a Claude call.

The exceptions:
- questions about commitments still go to a human;
- an answer whose policy section has changed version since it was approved isn't reused.

Peek at the library at any time:

In [ ]:
pd.read_sql("SELECT id, question, decision, reviewer, approved_at, times_reused, sources FROM answers ORDER BY id", lib)